# Características textuales interpretables y ranking pairwise

En este notebook se estudia si algunas características sencillas e interpretables de las respuestas están relacionadas con la preferencia observada mediante upvotes.

El análisis parte de la observación de que una heurística simple, como seleccionar la respuesta más larga, obtiene un rendimiento relevante. A partir de esta idea se estudian otras señales textuales, como la presencia de términos relacionados con ayuda profesional y determinadas expresiones de apoyo o empatía.

Primero se evalúan estas características individualmente como heurísticas y, posteriormente, se combinan mediante una regresión logística con planteamiento pairwise.

Se mantienen las mismas particiones train-test y los mismos folds utilizados en los experimentos con BERT y ModernBERT.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

In [2]:
url = (
    "https://huggingface.co/datasets/nbertagnolli/counsel-chat/"
    "raw/main/20220401_counsel_chat.csv"
)

df = pd.read_csv(url)

print("Preguntas totales:", df["questionID"].nunique())
print("Respuestas totales:", len(df))

Preguntas totales: 940
Respuestas totales: 2775


#### Preparación de los datos

Se conservan únicamente las preguntas con al menos dos respuestas candidatas y con alguna señal de preferencia, es decir, aquellas en las que al menos una respuesta tiene algún upvote.

In [3]:
df_modelo = df.copy()

# Preguntas con al menos dos respuestas
n_respuestas = (
    df_modelo.groupby("questionID")["answerText"]
    .transform("size")
)

df_modelo = df_modelo[
    n_respuestas >= 2
].copy()

# Preguntas con alguna señal de preferencia
max_upvotes = (
    df_modelo.groupby("questionID")["upvotes"]
    .transform("max")
)

df_modelo = df_modelo[
    max_upvotes > 0
].copy()

# Target: respuesta con máximo número de upvotes
df_modelo["target"] = (
    df_modelo["upvotes"]
    == df_modelo.groupby("questionID")["upvotes"]
    .transform("max")
).astype(int)

print(
    "Preguntas incluidas:",
    df_modelo["questionID"].nunique()
)

print(
    "Respuestas incluidas:",
    len(df_modelo)
)

Preguntas incluidas: 276
Respuestas incluidas: 1732


#### Caracterización de las preguntas

Para construir particiones más comparables, se resume la información a nivel de pregunta.
Para cada questionID se calcula:

- El número de respuestas candidatas.
- Cuántas respuestas comparten el máximo número de upvotes.
- si existe o no empate en la respuesta top.

In [4]:
info_preguntas = (
    df_modelo.groupby("questionID")
    .agg(
        n_respuestas=("answerText", "size")
    )
    .reset_index()
)

n_top = (
    df_modelo.assign(
        es_top=lambda x:
            x["upvotes"]
            == x.groupby("questionID")["upvotes"].transform("max")
    )
    .groupby("questionID")["es_top"]
    .sum()
    .reset_index(name="n_respuestas_top")
)

info_preguntas = info_preguntas.merge(
    n_top,
    on="questionID"
)

info_preguntas["empate"] = (
    info_preguntas["n_respuestas_top"] > 1
)

display(info_preguntas.head())

,questionID,n_respuestas,n_respuestas_top,empate
0,0,86,1,False
1,1,5,1,False
2,2,2,1,False
3,3,3,1,False
4,4,6,1,False


Comprobación rápida para detectar si algo salió raro:

In [5]:
print("Preguntas:", len(info_preguntas))
print(
    "Preguntas con empate:",
    info_preguntas["empate"].sum()
)
print(
    "Porcentaje con empate:",
    f"{info_preguntas['empate'].mean():.2%}"
)

Preguntas: 276
Preguntas con empate: 62
Porcentaje con empate: 22.46%


#### Definición de los estratos

Las preguntas se agrupan según dos características:

- El número de respuestas candidatas.
- La presencia o ausencia de empate en el máximo número de upvotes.

Para el número de respuestas se utilizan los intervalos 2, 3, 4-5, 6-10 y >10.
La combinación de ambas características define el estrato de cada pregunta.
Estos estratos se utilizarán posteriormente para intentar mantener una composición similar entre train, test y los folds de validación.

In [6]:
def grupo_n_respuestas(n):
    if n == 2:
        return "2"
    elif n == 3:
        return "3"
    elif n <= 5:
        return "4-5"
    elif n <= 10:
        return "6-10"
    else:
        return ">10"


info_preguntas["grupo_respuestas"] = (
    info_preguntas["n_respuestas"]
    .apply(grupo_n_respuestas)
)

info_preguntas["estrato"] = (
    info_preguntas["grupo_respuestas"]
    + ", empate="
    + info_preguntas["empate"].astype(str)
)

Se comprueba el número de preguntas de cada estrato antes de realizar las divisiones. Esto permite verificar que los grupos contienen suficientes ejemplos para poder distribuirlos entre las distintas particiones.

In [7]:
distribucion_estratos = (
    info_preguntas["estrato"]
    .value_counts()
    .sort_index()
)

display(
    distribucion_estratos.to_frame(
        "Número de preguntas"
    )
)

,Número de preguntas
estrato,
"2, empate=False",82
"2, empate=True",16
"3, empate=False",42
"3, empate=True",10
"4-5, empate=False",35
"4-5, empate=True",10
"6-10, empate=False",34
"6-10, empate=True",14
">10, empate=False",21


## Nuevo split train-test estratificado

Anteriormente, el split 80/20 se realizaba aleatoriamente por questionID. En esta versión se mantiene el mismo porcentaje, pero se realiza una división estratificada utilizando los estratos definidos anteriormente.
De esta forma se intenta mantener en train y test proporciones similares de:

- Preguntas con y sin empate.
- Preguntas con distinto número de respuestas candidatas.

La división continúa realizándose a nivel de questionID, por lo que todas las respuestas de una misma pregunta permanecen siempre en el mismo conjunto.

In [8]:
from sklearn.model_selection import train_test_split

preguntas_train, preguntas_test = train_test_split(
    info_preguntas,
    test_size=0.20,
    random_state=27,
    stratify=info_preguntas["estrato"]
)

Una vez asignadas las preguntas a train y test, se recuperan del dataset original todas las respuestas correspondientes a esas preguntas. Así, cada pregunta mantiene juntas todas sus respuestas dentro de una única partición.

In [9]:
train_equilibrado = df_modelo[
    df_modelo["questionID"].isin(
        preguntas_train["questionID"]
    )
].copy()

test_equilibrado = df_modelo[
    df_modelo["questionID"].isin(
        preguntas_test["questionID"]
    )
].copy()

print(
    "Preguntas en train:",
    train_equilibrado["questionID"].nunique()
)

print(
    "Preguntas en test:",
    test_equilibrado["questionID"].nunique()
)

Preguntas en train: 220
Preguntas en test: 56


In [10]:
preguntas_comunes = (
    set(train_equilibrado["questionID"])
    & set(test_equilibrado["questionID"])
)

print(
    "Preguntas compartidas entre train y test:",
    len(preguntas_comunes)
)

Preguntas compartidas entre train y test: 0


No existe solapamiento de preguntas entre train y test, por lo que todas las respuestas asociadas a un mismo questionID permanecen dentro de una única partición.

## Construcción de los 5 folds estratificados

Sobre el nuevo conjunto de train se construyen 5 folds de validación.
Es un cambio respecto al notebook anterior, donde se utilizaba GroupKFold únicamente para mantener juntas las respuestas de cada pregunta.
En esta versión, la división se realiza primero a nivel de pregunta mediante StratifiedKFold, utilizando el estrato como criterio. De esta forma, además de mantener las preguntas separadas, se intenta que cada fold tenga una proporción similar de empates y de números de respuestas candidatas.

In [11]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=27
)

folds_preguntas = list(
    skf.split(
        preguntas_train,
        preguntas_train["estrato"]
    )
)

print(
    "Número de folds:",
    len(folds_preguntas)
)

Número de folds: 5


Los 5 folds ya están definidos por pregunta. Ahora se localizan en train_equilibrado todas las respuestas correspondientes a las preguntas de entrenamiento y validación de cada fold.

In [12]:
cv_splits_equilibrados = []

for train_q_idx, val_q_idx in folds_preguntas:

    ids_train_fold = set(
        preguntas_train.iloc[
            train_q_idx
        ]["questionID"]
    )

    ids_val_fold = set(
        preguntas_train.iloc[
            val_q_idx
        ]["questionID"]
    )

    train_idx = np.where(
        train_equilibrado["questionID"]
        .isin(ids_train_fold)
    )[0]

    val_idx = np.where(
        train_equilibrado["questionID"]
        .isin(ids_val_fold)
    )[0]

    cv_splits_equilibrados.append(
        (train_idx, val_idx)
    )

Comprobación:

In [13]:
for fold, (train_idx, val_idx) in enumerate(
    cv_splits_equilibrados,
    start=1
):

    train_fold = train_equilibrado.iloc[
        train_idx
    ]

    val_fold = train_equilibrado.iloc[
        val_idx
    ]

    comunes = (
        set(train_fold["questionID"])
        & set(val_fold["questionID"])
    )

    print(
        f"Fold {fold}: "
        f"{train_fold['questionID'].nunique()} preguntas train | "
        f"{val_fold['questionID'].nunique()} preguntas validación | "
        f"{len(comunes)} compartidas"
    )

Fold 1: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 2: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 3: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 4: 176 preguntas train | 44 preguntas validación | 0 compartidas
Fold 5: 176 preguntas train | 44 preguntas validación | 0 compartidas


Se comprueba que en los 5 folds hay 0 preguntas compartidas entre entrenamiento y validación.

## Comprobación de las nuevas particiones

Antes de repetir el entrenamiento de BERT, se analiza la composición de los nuevos folds de validación y del conjunto de test.
Se comprueba:
- El número de preguntas.
- La proporción de preguntas con empate.
- El número medio y mediano de respuestas candidatas.
- La distribución del número de respuestas por pregunta.

El objetivo es verificar que las nuevas particiones sean aproximadamente comparables antes de utilizar sus resultados para evaluar el modelo.

In [14]:
def resumen_particion(info, nombre):
    return {
        "Partición": nombre,
        "Preguntas": len(info),
        "Preguntas con empate": info["empate"].sum(),
        "% con empate": info["empate"].mean() * 100,
        "Media respuestas": info["n_respuestas"].mean(),
        "Mediana respuestas": info["n_respuestas"].median(),
        "Mín. respuestas": info["n_respuestas"].min(),
        "Máx. respuestas": info["n_respuestas"].max()
    }

In [15]:
resumen_nuevo = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    resumen_nuevo.append(
        resumen_particion(
            info_val,
            f"Fold {fold}"
        )
    )

resumen_nuevo.append(
    resumen_particion(
        preguntas_test,
        "Test"
    )
)

resumen_nuevo = pd.DataFrame(
    resumen_nuevo
)

display(
    resumen_nuevo.round(2)
)

,Partición,Preguntas,Preguntas con empate,% con empate,Media respuestas,Mediana respuestas,Mín. respuestas,Máx. respuestas
0,Fold 1,44,10,22.73,6.11,3.0,2,51
1,Fold 2,44,9,20.45,5.25,3.0,2,28
2,Fold 3,44,9,20.45,5.52,3.0,2,31
3,Fold 4,44,11,25.00,8.61,3.0,2,107
4,Fold 5,44,11,25.00,6.93,3.0,2,86
5,Test,56,12,21.43,5.45,3.0,2,35


**Número de preguntas:** Los cinco folds tienen exactamente 44 preguntas cada uno. Test tiene 56, lo cual es normal: cada fold corresponde aproximadamente a una quinta parte de train, mientras que test es el 20 % del dataset completo. Aquí ya no hay el pequeño desequilibrio anterior de 40–44 preguntas entre folds.

**Preguntas con empate:** Esto están mucho más laineadas. Antes los folds iban desde 14,63 % hasta 30,95 %, mientras que test tenía 25,93 %. Ahora todos están aproximadamente en el mismo rango.



#### Distribución del número de respuestas

In [16]:
distribuciones_nuevas = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    dist = (
        info_val["grupo_respuestas"]
        .value_counts(normalize=True)
        .mul(100)
    )

    dist.name = f"Fold {fold}"
    distribuciones_nuevas.append(dist)


dist_test = (
    preguntas_test["grupo_respuestas"]
    .value_counts(normalize=True)
    .mul(100)
)

dist_test.name = "Test"

distribuciones_nuevas.append(
    dist_test
)

tabla_distribucion = pd.concat(
    distribuciones_nuevas,
    axis=1
).fillna(0)

display(
    tabla_distribucion
    .reindex(
        ["2", "3", "4-5", "6-10", ">10"]
    )
    .round(2)
)

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Test
grupo_respuestas,,,,,,
2,36.36,34.09,34.09,36.36,36.36,35.71
3,18.18,20.45,20.45,18.18,15.91,19.64
4-5,18.18,15.91,15.91,15.91,15.91,16.07
6-10,15.91,18.18,18.18,15.91,18.18,17.86
>10,11.36,11.36,11.36,13.64,13.64,10.71


La distribución del número de respuestas también ha mejorado mucho.

Por ejemplo, en preguntas con solo 2 respuestas:
- folds: 34,09–36,36 %
- test: 35,71 %

Con 4–5 respuestas:
- folds: 15,91–18,18 %
- test: 16,07 %

Con 6–10 respuestas:
- folds: 15,91–18,18 %
- test: 17,86 %

Y con más de 10 respuestas:
- folds: 11,36–13,64 %
- test: 10,71 %

Esto es especialmente importante porque antes test solo tenía 5,56 % de preguntas con más de 10 respuestas, mientras que los folds tenían entre 12,20 % y 17,07 %. Esa diferencia prácticamente ha desaparecido.

#### Distribución de los estratos

Esta comprobación es nueva y útil, porque permite ver conjuntamente las dos variables (número de respuestas candidatasy presencia o ausencia de empate en el máximo de upvotes) que hemos utilizado para equilibrar.

In [17]:
distribuciones_estratos = []

for fold, (_, val_q_idx) in enumerate(
    folds_preguntas,
    start=1
):

    info_val = preguntas_train.iloc[
        val_q_idx
    ]

    dist = (
        info_val["estrato"]
        .value_counts(normalize=True)
        .mul(100)
    )

    dist.name = f"Fold {fold}"
    distribuciones_estratos.append(dist)


dist_test_estratos = (
    preguntas_test["estrato"]
    .value_counts(normalize=True)
    .mul(100)
)

dist_test_estratos.name = "Test"

distribuciones_estratos.append(
    dist_test_estratos
)

tabla_estratos = pd.concat(
    distribuciones_estratos,
    axis=1
).fillna(0)

display(
    tabla_estratos
    .sort_index()
    .round(2)
)

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Test
estrato,,,,,,
"2, empate=False",29.55,29.55,29.55,29.55,29.55,30.36
"2, empate=True",6.82,4.55,4.55,6.82,6.82,5.36
"3, empate=False",15.91,15.91,15.91,13.64,13.64,16.07
"3, empate=True",2.27,4.55,4.55,4.55,2.27,3.57
"4-5, empate=False",13.64,13.64,13.64,11.36,11.36,12.50
"4-5, empate=True",4.55,2.27,2.27,4.55,4.55,3.57
"6-10, empate=False",11.36,13.64,13.64,11.36,11.36,12.50
"6-10, empate=True",4.55,4.55,4.55,4.55,6.82,5.36
">10, empate=False",6.82,6.82,6.82,9.09,9.09,7.14


La distribución de los estratos es muy similar entre los cinco folds y test. Cada estrato combina el número de respuestas candidatas y la presencia o ausencia de empate, y sus proporciones se mantienen bastante próximas entre las distintas particiones.
Las pequeñas diferencias observadas son esperables debido al número limitado de preguntas en algunos estratos. En conjunto, la estratificación ha permitido obtener particiones mucho más comparables, por lo que resultan más adecuadas para repetir la validación cruzada de BERT.

## Características textuales

Se definen algunas características sencillas de cada respuesta:

- Longitud de la respuesta.
- Frecuencia de las palabras therapy y counselor.
- Frecuencia de algunas expresiones explícitas de apoyo o empatía.

Estas características se han elegido por ser fácilmente interpretables y por tener sentido dentro del dominio estudiado.

El conjunto de test no se utiliza para seleccionar las características ni para tomar decisiones sobre el modelo.

#### Definición de las características

In [23]:
import re

EXPRESIONES_EMPATICAS = [
    "i understand",
    "i'm sorry",
    "i am sorry",
    "it sounds like",
    "that sounds",
    "i hear you",
    "you are not alone",
    "you're not alone",
    "it makes sense",
    "thank you for sharing"
]

PALABRAS_APOYO = [
    "help",
    "support",
    "hope"
]


def contar_expresiones(texto, expresiones):
    texto = str(texto).lower()

    return sum(
        texto.count(expresion)
        for expresion in expresiones
    )


def añadir_caracteristicas_textuales(datos):
    datos = datos.copy()

    texto = (
        datos["answerText"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    # Longitud de la respuesta
    datos["longitud"] = texto.str.len()

    # Palabras relacionadas con ayuda profesional
    datos["freq_therapy"] = texto.str.count(
        r"\btherapy\b"
    )

    datos["freq_counselor"] = texto.str.count(
        r"\bcounselor\b"
    )

    datos["freq_ayuda_profesional"] = (
        datos["freq_therapy"]
        + datos["freq_counselor"]
    )

    # Expresiones de apoyo o empatía
    datos["freq_empatia"] = (
        texto.apply(
            lambda x: contar_expresiones(
                x,
                EXPRESIONES_EMPATICAS
            )
        )
    )

    return datos

Las características se calculan tanto para train como para test mediante reglas fijadas previamente. El análisis y la selección del modelo se realizan utilizando únicamente el conjunto de entrenamiento y sus folds de validación.

In [19]:
train_caracteristicas = añadir_caracteristicas_textuales(
    train_equilibrado
)

test_caracteristicas = añadir_caracteristicas_textuales(
    test_equilibrado
)

train_caracteristicas[
    [
        "answerText",
        "longitud",
        "freq_therapy",
        "freq_counselor",
        "freq_ayuda_profesional",
        "freq_empatia"
    ]
].head()

,answerText,longitud,freq_therapy,freq_counselor,freq_ayuda_profesional,freq_empatia
0,It is very common for people to have multiple ...,982,0,0,0,0
1,"I've never heard of someone having ""too many i...",691,2,0,2,0
2,Absolutely not. I strongly recommending worki...,230,1,0,1,0
3,Let me start by saying there are never too man...,1512,0,0,0,0
4,I just want to acknowledge you for the courage...,1077,1,0,1,0


## Análisis de las características

Antes de utilizarlas como variables predictivas, se compara su distribución entre respuestas top y no top dentro del conjunto de entrenamiento.

Este análisis es descriptivo y permite comprobar si existen diferencias que puedan justificar su uso posterior como heurísticas o variables de un modelo.

In [20]:
CARACTERISTICAS = [
    "longitud",
    "freq_therapy",
    "freq_counselor",
    "freq_ayuda_profesional",
    "freq_empatia"
]

resumen_caracteristicas = (
    train_caracteristicas
    .groupby("target")[CARACTERISTICAS]
    .agg(["mean", "median"])
)

display(
    resumen_caracteristicas.round(2)
)

longitud        freq_therapy        freq_counselor         \
           mean median         mean median           mean median   
target                                                             
0        882.49  703.5         0.51    0.0           0.24    0.0   
1       1082.95  915.0         0.27    0.0           0.18    0.0   

       freq_ayuda_profesional        freq_empatia         
                         mean median         mean median  
target                                                    
0                        0.75    0.0         0.15    0.0  
1                        0.44    0.0         0.19    0.0

Las respuestas top presentan una mayor longitud media y mediana que las no top (1082,95 vs. 882,49 caracteres de media), reforzando la idea de que la longitud es una señal relevante.
En cambio, las menciones a therapy y counselor son, en promedio, algo más frecuentes en las respuestas no top. Además, la mediana es 0 en ambos grupos, lo que indica que estas palabras no aparecen en la mayoría de las respuestas.
Las expresiones de empatía presentan una diferencia pequeña a favor de las respuestas top (0,19 vs. 0,15), aunque también tienen mediana 0.

### Exploración de palabras asociadas a respuestas top

Se analiza, únicamente sobre el conjunto de entrenamiento, qué palabras aparecen con distinta frecuencia en respuestas top y no top. Este análisis tiene un carácter exploratorio y no se utiliza para modificar las características evaluadas en este experimento.

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    stop_words="english",
    min_df=10,
    binary=True
)

X = vectorizer.fit_transform(
    train_caracteristicas["answerText"].fillna("")
)

palabras = vectorizer.get_feature_names_out()

top = train_caracteristicas["target"].values == 1
no_top = ~top

freq_top = np.asarray(
    X[top].mean(axis=0)
).ravel()

freq_no_top = np.asarray(
    X[no_top].mean(axis=0)
).ravel()

analisis_palabras = pd.DataFrame({
    "palabra": palabras,
    "presencia_top": freq_top,
    "presencia_no_top": freq_no_top,
    "diferencia": freq_top - freq_no_top
})

display(
    analisis_palabras
    .sort_values("diferencia", ascending=False)
    .head(20)
)

,palabra,presencia_top,presencia_no_top,diferencia
601,help,0.481100,0.398768,0.082332
1034,really,0.202749,0.121479,0.081270
551,good,0.333333,0.263204,0.070129
774,love,0.154639,0.089789,0.064850
727,learn,0.158076,0.093310,0.064766
982,professional,0.137457,0.073063,0.064394
1059,relationship,0.285223,0.220951,0.064273
1389,way,0.326460,0.263204,0.063256
1253,support,0.192440,0.130282,0.062158
1332,try,0.199313,0.138204,0.061108


In [22]:
display(
    analisis_palabras
    .sort_values("diferencia")
    .head(20)
)

,palabra,presencia_top,presencia_no_top,diferencia
1291,therapy,0.175258,0.264085,-0.088827
1289,therapist,0.288660,0.346831,-0.058171
1142,session,0.017182,0.073944,-0.056762
980,process,0.092784,0.142606,-0.049822
290,crying,0.013746,0.058099,-0.044353
1288,therapeutic,0.010309,0.051056,-0.040747
1143,sessions,0.027491,0.066021,-0.038530
268,counseling,0.123711,0.159331,-0.035620
1195,space,0.020619,0.054577,-0.033959
212,comfortable,0.054983,0.088908,-0.033926


El análisis muestra que algunas palabras relacionadas con apoyo, como help, support y hope, aparecen con mayor frecuencia en las respuestas top. En cambio, términos directamente relacionados con terapia, como therapy, therapist y counseling, aparecen ligeramente más en las respuestas no top. Estos resultados motivan evaluar por separado señales relacionadas con apoyo general y con recomendación de ayuda profesional.

## Evaluación de heurísticas individuales

Se evalúa cada característica como una regla simple de selección top-1.

Para cada pregunta se selecciona la respuesta con el valor máximo de la característica considerada.

Cuando varias respuestas empatan en el valor máximo, la puntuación de esa pregunta corresponde a la probabilidad de seleccionar una respuesta correcta si se eligiera aleatoriamente entre las respuestas empatadas. De esta forma se evita favorecer artificialmente a las heurísticas que producen muchos empates.

In [21]:
def evaluar_heuristica(datos, columna):
    resultados = []
    empates = []

    for _, grupo in datos.groupby("questionID"):

        maximo = grupo[columna].max()

        candidatas = grupo[
            grupo[columna] == maximo
        ]

        # Accuracy esperada si hay empate
        resultados.append(
            candidatas["target"].mean()
        )

        empates.append(
            len(candidatas) > 1
        )

    return (
        np.mean(resultados),
        np.mean(empates)
    )

### Validación cruzada de las heurísticas

Las heurísticas se evalúan sobre los mismos cinco folds utilizados en los experimentos anteriores.

Aunque estas reglas no requieren entrenamiento, utilizar los mismos folds permite obtener una estimación comparable del rendimiento y de su variabilidad.

In [22]:
HEURISTICAS = {
    "Respuesta más larga": "longitud",
    "Más menciones a therapy": "freq_therapy",
    "Más menciones a counselor": "freq_counselor",
    "Más referencias a ayuda profesional": "freq_ayuda_profesional",
    "Más expresiones de empatía": "freq_empatia"
}

resultados_heuristicas_cv = []

for nombre, columna in HEURISTICAS.items():

    accuracies = []
    tasas_empate = []

    for _, val_idx in cv_splits_equilibrados:

        val_fold = (
            train_caracteristicas
            .iloc[val_idx]
            .copy()
        )

        accuracy, tasa_empate = (
            evaluar_heuristica(
                val_fold,
                columna
            )
        )

        accuracies.append(accuracy)
        tasas_empate.append(tasa_empate)

    resultados_heuristicas_cv.append({
        "Heurística": nombre,
        "Accuracy@1 media (%)":
            np.mean(accuracies) * 100,
        "Desv. estándar (%)":
            np.std(accuracies) * 100,
        "Preguntas con empate (%)":
            np.mean(tasas_empate) * 100
    })

resultados_heuristicas_cv = pd.DataFrame(
    resultados_heuristicas_cv
)

display(
    resultados_heuristicas_cv.round(2)
)

,Heurística,Accuracy@1 media (%),Desv. estándar (%),Preguntas con empate (%)
0,Respuesta más larga,46.36,6.20,0.00
1,Más menciones a therapy,39.39,1.92,69.09
2,Más menciones a counselor,41.45,2.59,75.00
3,Más referencias a ayuda profesional,42.51,1.65,60.45
4,Más expresiones de empatía,39.14,1.99,71.36


La longitud vuelve a ser la heurística con mejor rendimiento, con una Accuracy@1 media del 46,36 %. Las heurísticas basadas en palabras relacionadas con ayuda profesional obtienen resultados inferiores y presentan una elevada proporción de empates, ya que estos términos no aparecen en muchas de las respuestas. La combinación de therapy y counselor alcanza un 42,51 %, mientras que la heurística basada en expresiones de empatía obtiene un 39,14 %.

## Regresión logística pairwise con características interpretables

A continuación, las características textuales se combinan mediante una regresión logística.

Se mantiene el mismo planteamiento pairwise utilizado con BERT y ModernBERT: para cada pregunta se comparan dos respuestas y el objetivo es predecir cuál de ellas recibió más upvotes.

Las parejas con el mismo número de upvotes se descartan, ya que no proporcionan una preferencia observada.

Como entrada del modelo se utiliza la diferencia entre las características de ambas respuestas. Por ejemplo, una diferencia positiva en longitud indica que la primera respuesta es más larga que la segunda.

Uso estas cuatro señales:

In [23]:
FEATURES = [
    "longitud",
    "freq_therapy",
    "freq_counselor",
    "freq_empatia"
]

#### Construcción de pares

In [24]:
from itertools import combinations


def crear_pares_caracteristicas(datos):
    pares = []

    for question_id, grupo in datos.groupby(
        "questionID"
    ):
        filas = list(
            grupo.iterrows()
        )

        for (_, respuesta_a), (_, respuesta_b) in combinations(
            filas,
            2
        ):
            if (
                respuesta_a["upvotes"]
                == respuesta_b["upvotes"]
            ):
                continue

            label = int(
                respuesta_a["upvotes"]
                > respuesta_b["upvotes"]
            )

            ejemplo = {
                f"diff_{feature}":
                    respuesta_a[feature]
                    - respuesta_b[feature]
                for feature in FEATURES
            }

            ejemplo["label"] = label

            pares.append(ejemplo)

            # Orden inverso
            ejemplo_inverso = {
                f"diff_{feature}":
                    respuesta_b[feature]
                    - respuesta_a[feature]
                for feature in FEATURES
            }

            ejemplo_inverso["label"] = (
                1 - label
            )

            pares.append(
                ejemplo_inverso
            )

    return pd.DataFrame(pares)

Esto hace lo mismo conceptualmente que BERT: Respuesta A vs Respuesta B, pero en vez de pasar los textos completos, pasa cosas como:
- A tiene 250 caracteres más que B.
- A menciona "therapy" una vez más.
- A tiene una expresión empática más.

#### Modelo

Las variables tienen escalas diferentes, por lo que se estandarizan antes de entrenar la regresión logística.

Se utiliza una configuración sencilla para mantener el modelo interpretable y evitar introducir ajustes innecesarios.

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


COLUMNAS_PAIRWISE = [
    f"diff_{feature}"
    for feature in FEATURES
]


def crear_modelo_logistico():
    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "logistic_regression",
            LogisticRegression(
                max_iter=1000,
                random_state=27
            )
        )
    ])

Apunte: No se necesita class_weight="balanced" porque al añadir ambos órdenes de cada pareja se genera un conjunto equilibrado entre las dos clases.

### Selección de la respuesta top-1

Para evaluar el modelo se utiliza el mismo planteamiento general que con los modelos basados en encoders.

Todas las respuestas de una pregunta se comparan por parejas. Cada comparación proporciona una probabilidad de preferencia para ambas respuestas.

Las respuestas acumulan victorias y probabilidades y, finalmente, se selecciona como top-1 aquella que obtiene más victorias. En caso de empate se utiliza la puntuación acumulada como criterio de desempate.

In [26]:
def evaluar_torneo_caracteristicas(
    modelo,
    datos
):
    victorias = {
        idx: 0.0
        for idx in datos.index
    }

    puntuaciones = {
        idx: 0.0
        for idx in datos.index
    }

    for _, grupo in datos.groupby(
        "questionID"
    ):

        indices = list(
            grupo.index
        )

        for idx_i, idx_j in combinations(
            indices,
            2
        ):
            respuesta_i = datos.loc[idx_i]
            respuesta_j = datos.loc[idx_j]

            diferencias = [
                respuesta_i[feature]
                - respuesta_j[feature]
                for feature in FEATURES
            ]

            X = pd.DataFrame(
                [diferencias],
                columns=COLUMNAS_PAIRWISE
            )

            prob_i = (
                modelo.predict_proba(X)[0, 1]
            )

            prob_j = 1 - prob_i

            puntuaciones[idx_i] += prob_i
            puntuaciones[idx_j] += prob_j

            if prob_i > prob_j:
                victorias[idx_i] += 1

            elif prob_j > prob_i:
                victorias[idx_j] += 1

            else:
                victorias[idx_i] += 0.5
                victorias[idx_j] += 0.5

    ganadores = []

    for question_id, grupo in datos.groupby(
        "questionID"
    ):
        grupo = grupo.copy()

        grupo["victorias"] = (
            grupo.index.map(victorias)
        )

        grupo["puntuacion"] = (
            grupo.index.map(puntuaciones)
        )

        ganador = (
            grupo.sort_values(
                [
                    "victorias",
                    "puntuacion"
                ],
                ascending=[
                    False,
                    False
                ]
            )
            .iloc[0]
        )

        ganadores.append(
            ganador
        )

    ganadores = pd.DataFrame(
        ganadores
    )

    accuracy = (
        ganadores["target"].mean()
    )

    return accuracy, ganadores

### Validación cruzada

La regresión logística se evalúa utilizando exactamente los mismos cinco folds empleados para BERT y ModernBERT.

En cada fold, el modelo se entrena únicamente con las preguntas correspondientes al conjunto de entrenamiento y se evalúa mediante selección top-1 sobre las preguntas de validación.

In [27]:
accuracies_lr_cv = []

for fold, (train_idx, val_idx) in enumerate(
    cv_splits_equilibrados,
    start=1
):
    print(
        f"\n===== FOLD {fold} ====="
    )

    train_fold = (
        train_caracteristicas
        .iloc[train_idx]
        .copy()
    )

    val_fold = (
        train_caracteristicas
        .iloc[val_idx]
        .copy()
    )

    pares_train = (
        crear_pares_caracteristicas(
            train_fold
        )
    )

    modelo = crear_modelo_logistico()

    modelo.fit(
        pares_train[COLUMNAS_PAIRWISE],
        pares_train["label"]
    )

    accuracy_fold, _ = (
        evaluar_torneo_caracteristicas(
            modelo,
            val_fold
        )
    )

    accuracies_lr_cv.append(
        accuracy_fold
    )

    print(
        f"Accuracy@1 fold {fold}: "
        f"{accuracy_fold:.2%}"
    )


===== FOLD 1 =====
Accuracy@1 fold 1: 43.18%

===== FOLD 2 =====
Accuracy@1 fold 2: 45.45%

===== FOLD 3 =====
Accuracy@1 fold 3: 54.55%

===== FOLD 4 =====
Accuracy@1 fold 4: 52.27%

===== FOLD 5 =====
Accuracy@1 fold 5: 36.36%


Resultado de CV:

In [28]:
print(
    "Accuracy@1 media CV:",
    f"{np.mean(accuracies_lr_cv):.2%}"
)

print(
    "Desviación estándar:",
    f"{np.std(accuracies_lr_cv):.2%}"
)

Accuracy@1 media CV: 46.36%
Desviación estándar: 6.52%


La regresión logística pairwise obtiene una Accuracy@1 media del 46,36 %, con una desviación estándar del 6,52 %. Este resultado es similar al de la heurística de seleccionar la respuesta más larga (46,36 %), por lo que, en validación cruzada, combinar las distintas características textuales no aporta una mejora clara respecto a utilizar únicamente la longitud. La variabilidad entre folds es además relativamente alta.

### Entrenamiento final

Tras la validación cruzada, se entrena la regresión logística utilizando todas las preguntas del conjunto de entrenamiento.

Una vez fijado el modelo, se realiza una única evaluación sobre el conjunto de test.

In [29]:
pares_train_final = (
    crear_pares_caracteristicas(
        train_caracteristicas
    )
)

modelo_lr_final = (
    crear_modelo_logistico()
)

modelo_lr_final.fit(
    pares_train_final[
        COLUMNAS_PAIRWISE
    ],
    pares_train_final["label"]
)

Pipeline(steps=[('scaler', StandardScaler()),
                ('logistic_regression',
                 LogisticRegression(max_iter=1000, random_state=27))])

### Importancia de las características

Los coeficientes de la regresión logística permiten estudiar qué características están asociadas con una mayor probabilidad de que una respuesta sea preferida frente a otra.

Un coeficiente positivo indica que un mayor valor de esa característica en la primera respuesta aumenta la probabilidad estimada de que esta sea la preferida.

In [30]:
coeficientes = (
    modelo_lr_final
    .named_steps[
        "logistic_regression"
    ]
    .coef_[0]
)

tabla_coeficientes = pd.DataFrame({
    "Característica": FEATURES,
    "Coeficiente": coeficientes
})

tabla_coeficientes = (
    tabla_coeficientes
    .sort_values(
        "Coeficiente",
        ascending=False
    )
)

display(
    tabla_coeficientes.round(3)
)

,Característica,Coeficiente
0,longitud,0.670
1,freq_therapy,-0.075
3,freq_empatia,-0.133
2,freq_counselor,-0.158


Apunte: No significa causalidad, solo asociación predictiva.

La longitud es claramente la característica con mayor peso positivo (0,670): respuestas más largas tienden a tener mayor probabilidad de ser preferidas.

En cambio, therapy, counselor y las expresiones de empatía tienen coeficientes pequeños y negativos, por lo que no parecen aportar una señal positiva clara en este modelo.

### Evaluación final en test

In [34]:
accuracy_lr_test, ganadores_lr = (
    evaluar_torneo_caracteristicas(
        modelo_lr_final,
        test_caracteristicas
    )
)

print(
    "Regresión logística pairwise "
    ", Accuracy@1 test:",
    f"{accuracy_lr_test:.2%}"
)

Regresión logística pairwise , Accuracy@1 test: 44.64%


En test, la regresión logística pairwise obtiene una Accuracy@1 del 44,64 %. Este resultado es superior al obtenido por BERT (37,50 %) y ModernBERT (33,93 %) sobre el mismo conjunto de test, lo que muestra que un modelo sencillo basado en características interpretables puede competir e incluso superar a modelos más complejos en este experimento.

Dado que en validación su rendimiento era 46,36 %, el resultado de test es además bastante coherente con lo observado durante la validación.

### Heurísticas en test

Ahora que el experimento está cerrado, evaluo también las heurísticas.

In [32]:
resultados_heuristicas_test = []

for nombre, columna in HEURISTICAS.items():

    accuracy, tasa_empate = (
        evaluar_heuristica(
            test_caracteristicas,
            columna
        )
    )

    resultados_heuristicas_test.append({
        "Heurística": nombre,
        "Accuracy@1 test (%)":
            accuracy * 100,
        "Preguntas con empate (%)":
            tasa_empate * 100
    })

resultados_heuristicas_test = pd.DataFrame(
    resultados_heuristicas_test
)

display(
    resultados_heuristicas_test.round(2)
)

,Heurística,Accuracy@1 test (%),Preguntas con empate (%)
0,Respuesta más larga,39.29,0.00
1,Más menciones a therapy,39.53,80.36
2,Más menciones a counselor,40.91,73.21
3,Más referencias a ayuda profesional,43.12,64.29
4,Más expresiones de empatía,32.66,67.86


En test, la heurística con mejor resultado es “más referencias a ayuda profesional” con un 43,12 % de Accuracy@1, seguida de counselor (40,91 %) y therapy (39,53 %). Sin embargo, estas heurísticas generan empates en una gran parte de las preguntas (64–80 %), lo que indica que esas palabras aparecen poco y tienen una capacidad limitada para distinguir entre respuestas. La respuesta más larga obtiene 39,29 % sin empates, mientras que la heurística de empatía presenta el peor resultado (32,66 %).

## Comparación de modelos

Finalmente, se compara la regresión logística basada en características interpretables con los modelos pairwise basados en encoders evaluados previamente.

Todos utilizan las mismas particiones train-test y los mismos folds de validación.

In [33]:
comparacion_modelos = pd.DataFrame({
    "Modelo": [
        "Regresión logística",
        "BERT",
        "ModernBERT"
    ],
    "Accuracy@1 CV (%)": [
        np.mean(
            accuracies_lr_cv
        ) * 100,
        44.09,
        45.91
    ],
    "Desv. estándar CV (%)": [
        np.std(
            accuracies_lr_cv
        ) * 100,
        5.10,
        3.34
    ],
    "Accuracy@1 test (%)": [
        accuracy_lr_test * 100,
        37.50,
        33.93
    ]
})

display(
    comparacion_modelos.round(2)
)

,Modelo,Accuracy@1 CV (%),Desv. estándar CV (%),Accuracy@1 test (%)
0,Regresión logística,46.36,6.52,44.64
1,BERT,44.09,5.10,37.50
2,ModernBERT,45.91,3.34,33.93


La regresión logística obtiene el mejor resultado global: 46,36 % en validación cruzada y 44,64 % en test. BERT y ModernBERT presentan resultados similares en validación, pero peores en test (37,50 % y 33,93 %, respectivamente).  
En este experimento, por tanto, el modelo sencillo basado en características interpretables generaliza mejor que los modelos basados en encoders y además ofrece una explicación más clara de qué señales está utilizando.